# Titanic Survival Analysis: Logistic Regression and Random Forests

**The question:** Is there a relationship between passenger characteristics (such as class, gender, title, and fare) and survival outcomes on the Titanic?

**The data:** The [Kaggle Titanic dataset](https://www.kaggle.com/competitions/titanic), covering passenger details like age, sex, passenger class, fare, and whether they survived.

**Rough plan:**
1. **Feature Engineering & Data Cleaning:** We handle missing data by filling missing ages based on passenger titles (like *Mr.*, *Mrs.*, or *Master*). We engineer new features including child indicators (`IsChild`, `WomanOrChild`) and total family size.
2. **Two Separate Feature Sets:** We build two separate feature sets for the models, as logistic regression requires normalized inputs while decision trees do not.
   * **Logistic Regression:** Categorical variables are encoded with one reference category omitted per group (as described in Section 3.3.1 of *An Introduction to Statistical Learning with Applications in Python* by James et al., 2023), preventing collinearity. Continuous features are $Z$-score standardized ($z = \frac{x-\mu}{\sigma}$), while binary indicator columns are left untouched.
   * **Tree Models:** Tree models do not require us to scale continuous features or drop any category columns.
3. **Parametric Modeling & Regularization Penalties:** We fit Logistic Regression via Maximum Likelihood Estimation (MLE) and systematically evaluate four regularization penalties ($L_1$, $L_2$, Elastic Net, and Unregularized).
4. **Non-Parametric Tree Models:** We implement a single recursive Decision Tree and a Random Forest ensemble directly using their derived mathematical formulas (see main document).
5. **Predictive Evaluation:** We use our trained models on unseen test data ($n = 418$) to evaluate test accuracy across different models.

## 1. Setup
We start by importing the core libraries we'll need: **pandas** and **numpy** for data manipulation.

In [1]:
import pandas as pd
import numpy as np

## 2. Loading the Data
We load the Kaggle Titanic `train.csv` and `test.csv` files. The training set includes the `Survived` label for model training, while the test set contains only passenger features that we will generate final predictions for.

In [2]:
# Load the raw train and test CSVs
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

### Inspect the data
First, we check the column types and which columns have missing values.

In [3]:
print("Train Dataset Info")
print(train.info())

print("\nTest Dataset Info")
print(test.info())

Train Dataset Info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB
None

Test Dataset Info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Passeng

In [4]:
print("\nTrain Data")
train.head()


Train Data


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
print("\nTest Data")
test.head()


Test Data


,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


## 3. Handling Missing Values
Several columns contain missing values that must be filled because logistic regression cannot process empty entries. We fill these gaps rather than dropping rows to avoid losing valuable training data.

For `Age`, instead of using a single overall median, we fill missing entries using the median age for each passenger title (for example, *Master* reliably indicates a young boy).

### A closer look at passenger titles
Before filling in missing values, we look at the honorific titles extracted from the `Name` column, such as *Mr.*, *Mrs.*, and *Master*. These titles provide a much stronger signal for age and survival than ticket class alone.

In [6]:
# Access the full passenger names (e.g., "Braund, Mr. Owen Harris")
# Pattern: capture one or more letters ([A-Za-z]+) right before a period (\.)
# expand=False returns a 1D Series instead of a DataFrame
train_titles = train['Name'].str.extract(r'([A-Za-z]+)\.', expand=False)
test_titles = test['Name'].str.extract(r'([A-Za-z]+)\.', expand=False)

# Count the frequency of each unique title to inspect common vs rare categories across datasets
print("\nUnique Titles Found in Train")
print(train_titles.value_counts())

print("\nUnique Titles Found in Test")
print(test_titles.value_counts())


Unique Titles Found in Train
Name
Mr          517
Miss        182
Mrs         125
Master       40
Dr            7
Rev           6
Mlle          2
Major         2
Col           2
Countess      1
Capt          1
Ms            1
Sir           1
Lady          1
Mme           1
Don           1
Jonkheer      1
Name: count, dtype: int64

Unique Titles Found in Test
Name
Mr        240
Miss       78
Mrs        72
Master     21
Col         2
Rev         2
Ms          1
Dr          1
Dona        1
Name: count, dtype: int64


### Extract and group passenger titles
We extract titles from the name column and group infrequent titles like *Dr*, *Rev*, *Col*, and *Major* into a single *Rare* category. We also map French titles such as *Mlle* to *Miss* and *Mme* to *Mrs*.

In [7]:
# Extract and clean passenger titles
for df in [train, test]:
    df['Title'] = df['Name'].str.extract(r'([A-Za-z]+)\.', expand=False)
    df['Title'] = df['Title'].replace(
        ['Dr', 'Rev', 'Col', 'Major', 'Countess', 'Sir', 'Jonkheer', 'Lady', 'Capt', 'Don', 'Dona', 'Ms'],
        'Rare'
    )
    df['Title'] = df['Title'].replace({'Mlle': 'Miss', 'Mme': 'Mrs'})

### Fill missing Age using Title
Now that every passenger has a `Title`, we fill in the missing `Age` values using the **median age per title**, learned from the training set only (to avoid leaking test-set information).

In [8]:
# Impute missing Age using the median age per Title, learned from TRAIN only
age_by_title = train.groupby('Title')['Age'].median()

print("Median Ages per Title (from Train)")
print(age_by_title)

for title in train['Title'].unique():
    train.loc[(train['Title'] == title) & (train['Age'].isnull()), 'Age'] = age_by_title[title]
    # Apply the exact same learned training median to the matching missing rows in the test set
    test.loc[(test['Title'] == title) & (test['Age'].isnull()), 'Age'] = age_by_title[title]

print("Missing Age in Train:", train['Age'].isnull().sum())
print("Missing Age in Test: ", test['Age'].isnull().sum())

Median Ages per Title (from Train)
Title
Master     3.5
Miss      21.0
Mr        30.0
Mrs       35.0
Rare      48.0
Name: Age, dtype: float64
Missing Age in Train: 0
Missing Age in Test:  0


### Exploring Cabin Deck and Survival
Before handling missing `Cabin` entries, we check whether a passenger's cabin location carries a predictive survival signal. The first letter of the `Cabin` code corresponds to the deck level of the ship.

In [9]:
# Extract the cabin letter (e.g., 'C85' -> 'C'), filling missing entries with 'Unknown'
cabin_letter = train['Cabin'].str[0].fillna('Unknown')

cabin_survival = train.groupby(cabin_letter)['Survived'].agg(
    Total_Passengers='count',
    Survivors='sum',
    Survival_Rate='mean'
).reset_index()

cabin_survival['Survival_Rate_Pct'] = (cabin_survival['Survival_Rate'] * 100).round(2).astype(str) + '%'
cabin_survival = cabin_survival.sort_values(by='Survival_Rate', ascending=False)

print("Survival Rate by Cabin (Train Set)")
print(cabin_survival[['Cabin', 'Total_Passengers', 'Survivors', 'Survival_Rate_Pct']].to_string(index=False))

Survival Rate by Cabin (Train Set)
  Cabin  Total_Passengers  Survivors Survival_Rate_Pct
      D                33         25            75.76%
      E                32         24             75.0%
      B                47         35            74.47%
      F                13          8            61.54%
      C                59         35            59.32%
      G                 4          2             50.0%
      A                15          7            46.67%
Unknown               687        206            29.99%
      T                 1          0              0.0%


### Fill in Cabin, Embarked & Fare
We simplify `Cabin` to just its first letter (deck) and label missing entries as `Unknown`. Then we patch the last couple of gaps: 2 `Embarked` entries in train (filled with the most common port) and 1 `Fare` entry in test (filled with the train median fare, again to avoid leakage).

In [10]:
# Simplify Cabin to just the first letter and fill missing values with 'Unknown'
for df in [train, test]:
    df['Cabin'] = df['Cabin'].str[0].fillna('Unknown')

# Fill missing Embarked in train with the most frequent port
train['Embarked'] = train['Embarked'].fillna(train['Embarked'].mode()[0])

# Fill missing Fare in test using the median Fare learned from the training set (avoids data leakage)
test['Fare'] = test['Fare'].fillna(train['Fare'].median())

print(f"Total missing values remaining in Train: {train.isnull().sum().sum()}")
print(f"Total missing values remaining in Test: {test.isnull().sum().sum()}")

Total missing values remaining in Train: 0
Total missing values remaining in Test: 0


## 4. Shared Feature Engineering

These features are useful to *both* model families, so we build them once, before the two tables diverge:

* **Ticket Group Size:** passengers sharing a ticket number (families, friends, or staff booked together) are bucketed into `Solo`, `Small_Group` (2-4 people), and `Large_Group` (5+).
* **Family Size:** `SibSp` + `Parch` + 1, bucketed into `Solo`, `Small_Family` (2-4), and `Large_Family` (5+).
* **Evacuation Priority:** `IsChild` (titled *Master* or under age 12) and `WomanOrChild`, reflecting the historical "women and children first" evacuation protocol.

In [11]:
# 1. Count ticket frequencies across the combined manifest and bucket into group sizes
ticket_counts = (train['Ticket'].tolist() + test['Ticket'].tolist())
count_dict = {}
for t in ticket_counts:
    count_dict[t] = count_dict.get(t, 0) + 1

def get_ticket_group(size):
    if size == 1:
        return 'Solo'
    elif 2 <= size <= 4:
        return 'Small_Group'
    else:
        return 'Large_Group'

for df in [train, test]:
    df['Ticket'] = df['Ticket'].map(count_dict).apply(get_ticket_group)

summary = train.groupby('Ticket').agg(
    Total_Passengers=('Survived', 'count'),
    Survivors=('Survived', 'sum'),
    Survival_Rate=('Survived', 'mean')
).reset_index()
summary['Survival_Rate'] = (summary['Survival_Rate'] * 100).round(2).astype(str) + '%'
print(summary.to_string(index=False))

     Ticket  Total_Passengers  Survivors Survival_Rate
Large_Group                84         21         25.0%
Small_Group               326        191        58.59%
       Solo               481        130        27.03%


In [12]:
# 2. Family size = SibSp + Parch + 1, bucketed
def get_family_group(size):
    if size == 1:
        return 'Solo'
    elif 2 <= size <= 4:
        return 'Small_Family'
    else:
        return 'Large_Family'

for df in [train, test]:
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['Family_Group'] = df['FamilySize'].apply(get_family_group)

family_group_summary = train.groupby('Family_Group').agg(
    Passengers=('Survived', 'count'),
    Survivors=('Survived', 'sum'),
    Survival_Rate=('Survived', 'mean')
).reset_index()
family_group_summary['Survival_Rate_Pct'] = (family_group_summary['Survival_Rate'] * 100).round(2).astype(str) + '%'
print(family_group_summary[['Family_Group', 'Passengers', 'Survivors', 'Survival_Rate_Pct']].to_string(index=False))

Family_Group  Passengers  Survivors Survival_Rate_Pct
Large_Family          62         10            16.13%
Small_Family         292        169            57.88%
        Solo         537        163            30.35%


In [13]:
# 3. Evacuation priority features
for df in [train, test]:
    df['IsChild'] = ((df['Title'] == 'Master') | (df['Age'] < 12)).astype(int)
    df['WomanOrChild'] = ((df['Sex'] == 'female') | (df['IsChild'] == 1)).astype(int)

print("WomanOrChild breakdown in Train:")
print(train['WomanOrChild'].value_counts())

WomanOrChild breakdown in Train:
WomanOrChild
0    536
1    355
Name: count, dtype: int64


### Dropping Unusable Columns
We drop only the free-text and identifier columns that provide no predictive value on their own: `PassengerId` (saved separately for the submission file) and `Name` (now fully captured by `Title`).

In [14]:
# Save test PassengerId for the final submission, then drop Name/PassengerId
test_passenger_ids = test['PassengerId']
train = train.drop(columns=['Name', 'PassengerId'], errors='ignore')
test = test.drop(columns=['Name', 'PassengerId'], errors='ignore')

print(train.dtypes)

Survived          int64
Pclass            int64
Sex              object
Age             float64
SibSp             int64
Parch             int64
Ticket           object
Fare            float64
Cabin            object
Embarked         object
Title            object
FamilySize        int64
Family_Group     object
IsChild           int64
WomanOrChild      int64
dtype: object


## 5. Exploratory Analysis: Survival Patterns & Feature Interactions

Before deciding how to encode features for each model, we check baseline survival rates across key demographics and their combinations to see which interactions matter:

* **`Pclass` $\times$ `Sex`:** Did the "women and children first" priority hold across all ticket classes?
* **`Pclass` $\times$ `Age`:** Did protection for young passengers vary across socioeconomic levels?

These patterns explain why logistic regression benefits from explicit features like `WomanOrChild` and `IsChild`, without requiring us to manually construct every possible interaction combination.

In [15]:
base_rate = train['Survived'].mean() * 100
print(f"--- Baseline Overall Survival Rate: {base_rate:.2f}% ---\n")

print("Survival by Sex:")
print(train.groupby('Sex')['Survived'].agg(['count', 'mean']).rename(columns={'mean': 'Survival_Rate'}))
print("-" * 40)

print("Survival by Pclass:")
print(train.groupby('Pclass')['Survived'].agg(['count', 'mean']).rename(columns={'mean': 'Survival_Rate'}))
print("-" * 40)

print("Survival by Pclass & Sex:")
print(pd.pivot_table(train, values='Survived', index='Pclass', columns='Sex', aggfunc=['count', 'mean']))
print("-" * 40)

temp_age_bin = pd.cut(
    train['Age'],
    bins=[0, 12, 18, 35, 60, 100],
    labels=['Child (0-12)', 'Teen (13-18)', 'Young Adult (19-35)', 'Adult (36-60)', 'Senior (60+)']
)
print("Survival by Age Bracket:")
print(train.groupby(temp_age_bin, observed=False)['Survived'].agg(['count', 'mean']).rename(columns={'mean': 'Survival_Rate'}))
print("-" * 40)

print("Survival by Pclass & Age Bracket:")
print(pd.pivot_table(train, values='Survived', index=temp_age_bin, columns='Pclass', aggfunc=['count', 'mean'], observed=False))

--- Baseline Overall Survival Rate: 38.38% ---

Survival by Sex:
        count  Survival_Rate
Sex                         
female    314       0.742038
male      577       0.188908
----------------------------------------
Survival by Pclass:
        count  Survival_Rate
Pclass                      
1         216       0.629630
2         184       0.472826
3         491       0.242363
----------------------------------------
Survival by Pclass & Sex:
        count           mean          
Sex    female male    female      male
Pclass                                
1          94  122  0.968085  0.368852
2          76  108  0.921053  0.157407
3         144  347  0.500000  0.135447
----------------------------------------
Survival by Age Bracket:
                     count  Survival_Rate
Age                                      
Child (0-12)            73       0.575342
Teen (13-18)            70       0.428571
Young Adult (19-35)    530       0.352830
Adult (36-60)          196       0.3

### Key Insight

Ticket class strongly altered gender and age advantages: 3rd-class women and children survived at noticeably lower rates than those in 1st and 2nd class. Using targeted features such as `WomanOrChild` and `IsChild` directly captures this relationship.

## 6. Feature Set A — Logistic Regression

Two rules guide this feature set:

1. **Avoid collinearity.** Following James et al. (2023, Section 3.3.1), for every categorical variable with $k$ categories, we keep $k - 1$ indicator columns and drop one baseline category. 
2. **Normalize continuous features.** `Pclass`, `Age`, `SibSp`, `Parch`, `Fare`, and `FamilySize` are standardized ($Z$-score) using the training set mean and standard deviation. Binary ($0/1$) indicators are left unscaled to preserve their direct interpretation.

In [16]:
train_lr = train.copy()
test_lr = test.copy()

for df in [train_lr, test_lr]:
    # Sex: keep only Female (baseline = male)
    df['Female'] = (df['Sex'] == 'female').astype(int)

    # Embarked: baseline = S
    df['Embarked_C'] = (df['Embarked'] == 'C').astype(int)
    df['Embarked_Q'] = (df['Embarked'] == 'Q').astype(int)

    # Title: baseline = Mr
    df['Title_Miss']   = (df['Title'] == 'Miss').astype(int)
    df['Title_Mrs']    = (df['Title'] == 'Mrs').astype(int)
    df['Title_Master'] = (df['Title'] == 'Master').astype(int)
    df['Title_Rare']   = (df['Title'] == 'Rare').astype(int)

    # Ticket group: baseline = Solo
    df['Ticket_Small'] = (df['Ticket'] == 'Small_Group').astype(int)
    df['Ticket_Large'] = (df['Ticket'] == 'Large_Group').astype(int)

    # Family group: baseline = Solo
    df['Family_Small'] = (df['Family_Group'] == 'Small_Family').astype(int)
    df['Family_Large'] = (df['Family_Group'] == 'Large_Family').astype(int)

    # Cabin deck: baseline = Unknown
    for deck in ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'T']:
        df[f'Cabin_{deck}'] = (df['Cabin'] == deck).astype(int)

# Drop the original string columns now that they're encoded
drop_original = ['Sex', 'Embarked', 'Title', 'Ticket', 'Cabin', 'Family_Group']
for df in [train_lr, test_lr]:
    df.drop(columns=[c for c in drop_original if c in df.columns], inplace=True)

print(f"Logistic regression feature table: {train_lr.shape[1]} columns (incl. Survived)")
print(train_lr.dtypes)

Logistic regression feature table: 28 columns (incl. Survived)
Survived          int64
Pclass            int64
Age             float64
SibSp             int64
Parch             int64
Fare            float64
FamilySize        int64
IsChild           int64
WomanOrChild      int64
Female            int64
Embarked_C        int64
Embarked_Q        int64
Title_Miss        int64
Title_Mrs         int64
Title_Master      int64
Title_Rare        int64
Ticket_Small      int64
Ticket_Large      int64
Family_Small      int64
Family_Large      int64
Cabin_A           int64
Cabin_B           int64
Cabin_C           int64
Cabin_D           int64
Cabin_E           int64
Cabin_F           int64
Cabin_G           int64
Cabin_T           int64
dtype: object


### Z-score standardize the continuous columns only

$$z = \frac{x - \mu}{\sigma}$$

$\mu_{\text{train}}$ and $\sigma_{\text{train}}$ are computed **from the training set** and reused to transform the test set.

In [17]:
continuous_cols = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize']

mu = train_lr[continuous_cols].mean()
sigma = train_lr[continuous_cols].std(ddof=0)
sigma = sigma.replace(0, 1e-15)  # guard against any constant column

train_lr[continuous_cols] = (train_lr[continuous_cols] - mu) / sigma
test_lr[continuous_cols] = (test_lr[continuous_cols] - mu) / sigma

print("--- Continuous features standardized; binary indicators left at 0/1 ---")
print("Train Mean of continuous features (should be ~0):\n", train_lr[continuous_cols].mean().round(2))
print("\nTrain Std of continuous features (should be 1):\n", train_lr[continuous_cols].std(ddof=0).round(2))

--- Continuous features standardized; binary indicators left at 0/1 ---
Train Mean of continuous features (should be ~0):
 Pclass       -0.0
Age           0.0
SibSp         0.0
Parch         0.0
Fare          0.0
FamilySize   -0.0
dtype: float64

Train Std of continuous features (should be 1):
 Pclass        1.0
Age           1.0
SibSp         1.0
Parch         1.0
Fare          1.0
FamilySize    1.0
dtype: float64


# 7. Model Implementations & Regularization Penalties — Logistic Regression

We train four logistic regression models on `train_lr` based on our theoretical derivations (see the main document with derivations):

* **Unregularized MLE:** Unconstrained baseline optimizing cross-entropy loss.
* **$L_2$ (Ridge):** Shrinks large weights smoothly via quadratic penalty ($\frac{\lambda}{2n}\sum w_j^2$).
* **$L_1$ (Lasso):** Sets redundant weights to zero via absolute penalty ($\frac{\lambda}{n}\sum \vert w_j \vert$).
* **Elastic Net:** Combines $L_1$ and $L_2$ penalties to get the benefits of both.

### Variant 1 of 4: No Penalty (Unregularized MLE)

We begin with standard Maximum Likelihood Estimation via gradient descent. The training loop computes predicted probabilities using `sigmoid` and updates weights using the unregularized loss gradient.

In [18]:
def sigmoid(z):
    z = np.clip(z, -500, 500)  # Prevent numerical overflow
    return 1.0 / (1.0 + np.exp(-z))

def train_logistic_regression_no_penalty(X, y, lr=0.01, epochs=1000):
    n_samples, n_features = X.shape
    w = np.zeros(n_features)
    b = 0.0

    for _ in range(epochs):
        z = X.dot(w) + b
        y_hat = sigmoid(z)

        # Gradients
        dw = (1.0 / n_samples) * X.T.dot(y_hat - y)
        db = np.mean(y_hat - y)

        # Parameter updates
        w -= lr * dw
        b -= lr * db

    return w, b

### Baseline Training, Evaluation & Test Predictions

We train on the standardized training set, evaluate training accuracy, and generate the final test predictions to save in `submission_logreg.csv`.

In [19]:
# 1. Separate Features (X) and Target (y)
lr_features = [col for col in train_lr.columns if col != 'Survived']

y_train = train_lr['Survived'].values
X_train = train_lr[lr_features].values
X_test  = test_lr[lr_features].values

# 2. Train the unregularized model (no loss_hist returned)
w_unreg, b_unreg = train_logistic_regression_no_penalty(
    X_train, y_train, lr=0.01, epochs=5000
)

# 3. Prediction function, reused by every later variant
def predict(X, w, b, threshold=0.5):
    z = X.dot(w) + b
    probabilities = sigmoid(z)
    return (probabilities >= threshold).astype(int)

# 4. Evaluate on TRAIN set
train_predictions = predict(X_train, w_unreg, b_unreg)
train_accuracy = np.mean(train_predictions == y_train) * 100
print(f"Training Accuracy: {train_accuracy:.2f}%")

# 5. Predict on TEST set and export submission CSV
test_predictions = predict(X_test, w_unreg, b_unreg)

submission = pd.DataFrame({
    'PassengerId': test_passenger_ids,
    'Survived': test_predictions
})
submission.to_csv('submission_logreg.csv', index=False)

print("\nPredictions exported to 'submission_logreg.csv'")
print(f"Total Test Passengers: {len(submission)}")
print(f"Predicted Survivors: {submission['Survived'].sum()}")
print("\nFirst 10 predictions:")
print(submission.head(10))

Training Accuracy: 82.83%

Predictions exported to 'submission_logreg.csv'
Total Test Passengers: 418
Predicted Survivors: 168

First 10 predictions:
   PassengerId  Survived
0          892         0
1          893         1
2          894         0
3          895         0
4          896         1
5          897         0
6          898         1
7          899         0
8          900         1
9          901         0


### Variant 2 of 4: L2 Regularization (Ridge)
We add an $L_2$ penalty term to both the gradient and the loss, shrinking all weights toward zero proportionally to their size.

In [20]:
def train_logistic_regression_l2(X, y, l2_ratio=0.1, lr=0.01, epochs=5000):
    n_samples, n_features = X.shape
    w = np.zeros(n_features)
    b = 0.0

    for _ in range(epochs):
        z = X.dot(w) + b
        y_hat = sigmoid(z)

        # Gradients (including L2 penalty gradient on weights)
        dw = (1.0 / n_samples) * X.T.dot(y_hat - y) + (l2_ratio / n_samples) * w
        db = np.mean(y_hat - y)

        # Parameter updates
        w -= lr * dw
        b -= lr * db

    return w, b

### Tune the L2 penalty strength (λ)
We split the training data 80/20 into train/validation, train a model per candidate $\lambda$, and pick whichever gives the highest validation accuracy, then retrain on the **full** training set using that $\lambda$.

In [21]:
# 1. Create an 80/20 Train-Validation Split
np.random.seed(42)
indices = np.random.permutation(len(X_train))
split_idx = int(len(X_train) * 0.8)
train_idx, val_idx = indices[:split_idx], indices[split_idx:]
X_tr, y_tr = X_train[train_idx], y_train[train_idx]
X_val, y_val = X_train[val_idx], y_train[val_idx]

# 2. Grid Search evaluating on VALIDATION Accuracy
def find_best_l2_by_validation(X_tr, y_tr, X_val, y_val, lambda_grid, lr=0.1, epochs=2000):
    best_lambda = None
    best_val_acc = -1.0
    print("--- Searching for Best L2 Lambda (By Validation Accuracy) ---")
    for l2_val in lambda_grid:
        w, b = train_logistic_regression_l2(X_tr, y_tr, l2_ratio=l2_val, lr=lr, epochs=epochs)
        val_preds = predict(X_val, w, b)
        val_acc = np.mean(val_preds == y_val) * 100
        print(f"Lambda: {l2_val:<8} | Val Accuracy: {val_acc:.2f}%")
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_lambda = l2_val
    print(f"\nBest Lambda found: {best_lambda} (Validation Accuracy: {best_val_acc:.2f}%)")
    return best_lambda

lambda_options = [0.0001, 0.001, 0.01, 0.1, 1.0]
best_l2_lambda = find_best_l2_by_validation(X_tr, y_tr, X_val, y_val, lambda_options, lr=0.1, epochs=2000)

# 3. Retrain on FULL X_train using the Best Lambda
w_l2_final, b_l2_final = train_logistic_regression_l2(
    X_train, y_train, l2_ratio=best_l2_lambda, lr=0.1, epochs=2000
)

--- Searching for Best L2 Lambda (By Validation Accuracy) ---
Lambda: 0.0001   | Val Accuracy: 83.80%
Lambda: 0.001    | Val Accuracy: 83.80%
Lambda: 0.01     | Val Accuracy: 83.80%
Lambda: 0.1      | Val Accuracy: 83.24%
Lambda: 1.0      | Val Accuracy: 83.80%

Best Lambda found: 0.0001 (Validation Accuracy: 83.80%)


### Evaluate and export the L2 model

In [22]:
train_predictions_l2 = predict(X_train, w_l2_final, b_l2_final)
train_accuracy_l2 = np.mean(train_predictions_l2 == y_train) * 100
print(f" L2 Model Training Accuracy: {train_accuracy_l2:.2f}%")

test_predictions_l2 = predict(X_test, w_l2_final, b_l2_final)

submission_l2 = pd.DataFrame({
    'PassengerId': test_passenger_ids,
    'Survived': test_predictions_l2
})
submission_l2.to_csv('submission_l2.csv', index=False)

print("\n Predictions successfully exported to 'submission_l2.csv'!")
print(f"Total Test Passengers: {len(submission_l2)}")
print(f"Predicted Survivors: {submission_l2['Survived'].sum()}")
print("\nFirst 10 predictions:")
print(submission_l2.head(10))

 L2 Model Training Accuracy: 83.73%

 Predictions successfully exported to 'submission_l2.csv'!
Total Test Passengers: 418
Predicted Survivors: 167

First 10 predictions:
   PassengerId  Survived
0          892         0
1          893         1
2          894         0
3          895         0
4          896         1
5          897         0
6          898         1
7          899         0
8          900         1
9          901         0


### Variant 3 of 4: $L_1$ Regularization (Lasso)

We add an $L_1$ penalty ($\frac{\lambda}{n}\|w\|_1$) using the soft-thresholding update rule:
$$w_j^{(t+1)} = \operatorname{sign}\left( z_j^{(t)} \right) \max\left( 0, \, \left| z_j^{(t)} \right| - \frac{\alpha \lambda}{n} \right)$$
Whenever the unregularized step magnitude $|z_j^{(t)}|$ falls below the threshold $\frac{\alpha \lambda}{n}$, the weight is set exactly to zero, performing automatic feature selection.

In [23]:
def train_logistic_regression_l1_proximal(X, y, l1_ratio=0.1, lr=0.01, epochs=5000):
    n_samples, n_features = X.shape
    w = np.zeros(n_features)
    b = 0.0
    threshold = lr * l1_ratio / n_samples  # alpha * lambda / n

    for _ in range(epochs):
        z_lin = X.dot(w) + b
        y_hat = sigmoid(z_lin)

        # 1. Unregularized gradient step
        dw_unreg = (1.0 / n_samples) * X.T.dot(y_hat - y)
        db = np.mean(y_hat - y)
        z = w - lr * dw_unreg

        # 2. Soft-thresholding (Proximal Operator)
        w = np.sign(z) * np.maximum(0.0, np.abs(z) - threshold)
        b -= lr * db

    return w, b

### Tune the L1 penalty strength (λ)

In [24]:
# 1. Create an 80/20 Train-Validation Split
np.random.seed(42)
indices = np.random.permutation(len(X_train))
split_idx = int(len(X_train) * 0.8)
train_idx, val_idx = indices[:split_idx], indices[split_idx:]
X_tr, y_tr = X_train[train_idx], y_train[train_idx]
X_val, y_val = X_train[val_idx], y_train[val_idx]

# 2. Grid Search evaluating on VALIDATION Accuracy
def find_best_l1_by_validation(X_tr, y_tr, X_val, y_val, lambda_grid, lr=0.1, epochs=2000):
    best_lambda = None
    best_val_acc = -1.0
    print("--- Searching for Best L1 Lambda (By Validation Accuracy) ---")
    for l1_val in lambda_grid:
        w, b = train_logistic_regression_l1_proximal(X_tr, y_tr, l1_ratio=l1_val, lr=lr, epochs=epochs)
        val_preds = predict(X_val, w, b)
        val_acc = np.mean(val_preds == y_val) * 100
        print(f"Lambda: {l1_val:<8} | Val Accuracy: {val_acc:.2f}%")
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_lambda = l1_val
    print(f"\nBest L1 Lambda found: {best_lambda} (Validation Accuracy: {best_val_acc:.2f}%)")
    return best_lambda

lambda_options = [0.0001, 0.001, 0.01, 0.1, 1.0]
best_l1_lambda = find_best_l1_by_validation(X_tr, y_tr, X_val, y_val, lambda_options, lr=0.1, epochs=2000)

# 3. Retrain on FULL X_train using the Best Lambda
w_l1, b_l1 = train_logistic_regression_l1_proximal(
    X_train, y_train, l1_ratio=best_l1_lambda, lr=0.1, epochs=2000
)

--- Searching for Best L1 Lambda (By Validation Accuracy) ---
Lambda: 0.0001   | Val Accuracy: 83.80%
Lambda: 0.001    | Val Accuracy: 83.80%
Lambda: 0.01     | Val Accuracy: 83.80%
Lambda: 0.1      | Val Accuracy: 83.24%
Lambda: 1.0      | Val Accuracy: 84.36%

Best L1 Lambda found: 1.0 (Validation Accuracy: 84.36%)


In [25]:
zero_indices = np.where(w_l1 == 0)[0]
non_zero_indices = np.where(w_l1 != 0)[0]

print(f"Total features: {len(w_l1)}")
print(f"Features set to exactly 0: {len(zero_indices)} ({len(zero_indices) / len(w_l1) * 100:.1f}%)")
print(f"Features kept: {len(non_zero_indices)}")
print(f"\nFeatures set to 0:")
print([lr_features[i] for i in zero_indices])

Total features: 27
Features set to exactly 0: 7 (25.9%)
Features kept: 20

Features set to 0:
['Title_Miss', 'Title_Rare', 'Ticket_Small', 'Cabin_A', 'Cabin_B', 'Cabin_G', 'Cabin_T']


### Evaluate and export the L1 model

In [26]:
train_predictions_l1 = predict(X_train, w_l1, b_l1)
train_accuracy_l1 = np.mean(train_predictions_l1 == y_train) * 100
print(f" L1 Model Training Accuracy: {train_accuracy_l1:.2f}%")

test_predictions_l1 = predict(X_test, w_l1, b_l1)

submission_l1 = pd.DataFrame({
    'PassengerId': test_passenger_ids,
    'Survived': test_predictions_l1
})
submission_l1.to_csv('submission_l1.csv', index=False)

print("\n Predictions successfully exported to 'submission_l1.csv'!")
print(f"Total Test Passengers: {len(submission_l1)}")
print(f"Predicted Survivors: {submission_l1['Survived'].sum()}")
print("\nFirst 10 predictions:")
print(submission_l1.head(10))

 L1 Model Training Accuracy: 83.39%

 Predictions successfully exported to 'submission_l1.csv'!
Total Test Passengers: 418
Predicted Survivors: 169

First 10 predictions:
   PassengerId  Survived
0          892         0
1          893         1
2          894         0
3          895         0
4          896         1
5          897         0
6          898         1
7          899         0
8          900         1
9          901         0


### Variant 4 of 4: Elastic Net (L1 + L2 combined)
Blends both penalties via separate `l1_ratio` and `l2_ratio` strengths.

In [27]:
def train_logistic_regression_elasticnet(X, y, l1_ratio=0.01, l2_ratio=0.01, lr=0.01, epochs=5000):
    n_samples, n_features = X.shape
    w = np.zeros(n_features)
    b = 0.0

    for _ in range(epochs):
        z = X.dot(w) + b
        y_hat = sigmoid(z)

        # Gradients (including both L1 subgradient and L2 gradient penalties)
        dw = (1.0 / n_samples) * X.T.dot(y_hat - y) + (l1_ratio / n_samples) * np.sign(w) + (l2_ratio / n_samples) * w
        db = np.mean(y_hat - y)

        # Parameter updates
        w -= lr * dw
        b -= lr * db

    return w, b

### Tune both penalty strengths (λ₁, λ₂)

In [28]:
# 1. Create an 80/20 Train-Validation Split
np.random.seed(42)
indices = np.random.permutation(len(X_train))
split_idx = int(len(X_train) * 0.8)
train_idx, val_idx = indices[:split_idx], indices[split_idx:]
X_tr, y_tr = X_train[train_idx], y_train[train_idx]
X_val, y_val = X_train[val_idx], y_train[val_idx]

# 2. Grid Search evaluating on VALIDATION Accuracy
def find_best_elasticnet_by_validation(X_tr, y_tr, X_val, y_val, grid_l1, grid_l2, lr=0.1, epochs=2000):
    best_val_acc = -1.0
    best_l1, best_l2 = None, None
    print("--- Searching for Best ElasticNet Parameters (By Validation Accuracy) ---")
    for l1_val in grid_l1:
        for l2_val in grid_l2:
            w, b = train_logistic_regression_elasticnet(
                X_tr, y_tr, l1_ratio=l1_val, l2_ratio=l2_val, lr=lr, epochs=epochs
            )
            val_preds = predict(X_val, w, b)
            val_acc = np.mean(val_preds == y_val) * 100
            print(f"L1: {l1_val:<6} | L2: {l2_val:<6} | Val Accuracy: {val_acc:.2f}%")
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_l1, best_l2 = l1_val, l2_val
    print(f"\nBest ElasticNet Found -> L1: {best_l1}, L2: {best_l2} (Val Accuracy: {best_val_acc:.2f}%)")
    return best_l1, best_l2

grid_l1_options = [0.0001, 0.01, 0.1]
grid_l2_options = [0.0001, 0.01, 0.1]

best_l1_en, best_l2_en = find_best_elasticnet_by_validation(
    X_tr, y_tr, X_val, y_val, grid_l1_options, grid_l2_options, lr=0.1, epochs=2000
)

# 3. Retrain on FULL X_train using Best L1 and L2 Parameters
w_en, b_en = train_logistic_regression_elasticnet(
    X_train, y_train, l1_ratio=best_l1_en, l2_ratio=best_l2_en, lr=0.1, epochs=2000
)

--- Searching for Best ElasticNet Parameters (By Validation Accuracy) ---
L1: 0.0001 | L2: 0.0001 | Val Accuracy: 83.80%
L1: 0.0001 | L2: 0.01   | Val Accuracy: 83.80%
L1: 0.0001 | L2: 0.1    | Val Accuracy: 83.24%
L1: 0.01   | L2: 0.0001 | Val Accuracy: 83.80%
L1: 0.01   | L2: 0.01   | Val Accuracy: 83.80%
L1: 0.01   | L2: 0.1    | Val Accuracy: 83.24%
L1: 0.1    | L2: 0.0001 | Val Accuracy: 83.24%
L1: 0.1    | L2: 0.01   | Val Accuracy: 83.24%
L1: 0.1    | L2: 0.1    | Val Accuracy: 83.24%

Best ElasticNet Found -> L1: 0.0001, L2: 0.0001 (Val Accuracy: 83.80%)


### Evaluate and export the Elastic Net model

In [29]:
train_predictions_en = predict(X_train, w_en, b_en)
train_accuracy_en = np.mean(train_predictions_en == y_train) * 100
print(f" ElasticNet Model Training Accuracy: {train_accuracy_en:.2f}%")

test_predictions_en = predict(X_test, w_en, b_en)

submission_en = pd.DataFrame({
    'PassengerId': test_passenger_ids,
    'Survived': test_predictions_en
})
submission_en.to_csv('submission_elasticnet.csv', index=False)

print("\n Predictions successfully exported to 'submission_elasticnet.csv'!")
print(f"Total Test Passengers: {len(submission_en)}")
print(f"Predicted Survivors: {submission_en['Survived'].sum()}")
print("\nFirst 10 predictions:")
print(submission_en.head(10))


 ElasticNet Model Training Accuracy: 83.73%

 Predictions successfully exported to 'submission_elasticnet.csv'!
Total Test Passengers: 418
Predicted Survivors: 167

First 10 predictions:
   PassengerId  Survived
0          892         0
1          893         1
2          894         0
3          895         0
4          896         1
5          897         0
6          898         1
7          899         0
8          900         1
9          901         0


## 8. Comparing the Four Logistic Regression Variants
With predictions from all four variants saved to CSV, we compare how many survivors each model predicts and how much the models agree with each other.

In [30]:
sub_unreg = pd.read_csv('submission_logreg.csv')
sub_l2 = pd.read_csv('submission_l2.csv')
sub_l1 = pd.read_csv('submission_l1.csv')
sub_en = pd.read_csv('submission_elasticnet.csv')

print("--- Model Comparison: Total Survivors Predicted ---")
print(f"Unregularized: {sub_unreg['Survived'].sum()} / {len(sub_unreg)}")
print(f"L2 (Ridge): {sub_l2['Survived'].sum()} / {len(sub_l2)}")
print(f"L1 (Lasso): {sub_l1['Survived'].sum()} / {len(sub_l1)}")
print(f"ElasticNet: {sub_en['Survived'].sum()} / {len(sub_en)}")

agree_l1_l2 = np.mean(sub_l1['Survived'] == sub_l2['Survived']) * 100
agree_l2_en = np.mean(sub_l2['Survived'] == sub_en['Survived']) * 100
agree_l2_unreg = np.mean(sub_l2['Survived'] == sub_unreg['Survived']) * 100

print("\n--- Agreement Check ---")
print(f"L1 and L2 predictions match: {agree_l1_l2:.2f}% of the time")
print(f"L2 and ElasticNet match: {agree_l2_en:.2f}% of the time")
print(f"L2 and Unregularized match: {agree_l2_unreg:.2f}% of the time")

--- Model Comparison: Total Survivors Predicted ---
Unregularized: 168 / 418
L2 (Ridge): 167 / 418
L1 (Lasso): 169 / 418
ElasticNet: 167 / 418

--- Agreement Check ---
L1 and L2 predictions match: 99.52% of the time
L2 and ElasticNet match: 100.00% of the time
L2 and Unregularized match: 98.33% of the time


### Feature Importance & Interpretation

Because all four models produce similar predictions and agreement rates, we examine the learned weights from the **unregularized model** to understand overall feature trends and their direct impact on survival.

In [31]:
importance_df = pd.DataFrame(
    {"Feature": lr_features, "Weight": w_unreg}
).sort_values(by="Weight", key=abs, ascending=False)

print("--- Logistic Regression (Unregularized): Top 10 Feature Weights ---")
importance_df.head(10)

--- Logistic Regression (Unregularized): Top 10 Feature Weights ---


,Feature,Weight
7,WomanOrChild,1.245151
8,Female,0.976541
0,Pclass,-0.762875
12,Title_Mrs,0.640198
16,Ticket_Large,-0.494180
18,Family_Large,-0.451763
1,Age,-0.401473
13,Title_Master,0.295586
11,Title_Miss,0.278106
23,Cabin_E,0.245063


## Summary: Logistic Regression

* **Penalty Convergence & Kaggle Benchmark:** All four penalties produced highly consistent results on the unseen test set, predicting between 167 and 169 survivors. On the Kaggle public leaderboard:
  * **Unregularized MLE:** Achieved **77.51%** accuracy (324/418 correct).
  * **L2 (Ridge) & ElasticNet:** Achieved **77.27%** accuracy (323/418 correct).
  * **L1 (Lasso):** Achieved **76.79%** accuracy (321/418 correct).
* **Weight Interpretability (Unregularized Model):** Because all four models yielded 98.3% to 100.0% prediction agreement, the unregularized weights cleanly capture the dominant linear relationships:
  * `WomanOrChild` ($w = +1.245$), `Female` ($w = +0.977$), and `Title_Mrs` ($w = +0.640$) provided the strongest positive contributions to survival log-odds.
  * `Pclass` ($w = -0.763$), `Ticket_Large` ($w = -0.494$), `Family_Large` ($w = -0.452$), and `Age` ($w = -0.401$) represented the heaviest penalties.
* **L1 Sparsity Results:** Proximal gradient descent with $\lambda = 1.0$ drove **7 out of 27 features (25.9%) strictly to 0.0**: `Title_Miss`, `Title_Rare`, `Ticket_Small`, `Cabin_A`, `Cabin_B`, `Cabin_G`, and `Cabin_T`. While this created a sparser model, it sacrificed a marginal degree of test accuracy on Kaggle (76.79% vs. 77.51%).

## 9. Feature Set B — Decision Tree / Random Forest

Tree models do not require the same preprocessing constraints as logistic regression:

* **No standardization:** Splits rely on inequality thresholds (`feature <= threshold`). Monotonic transformations do not alter split decisions, so `Age`, `Fare`, and `Pclass` remain in their original units.
* **Retaining all category indicators:** Because the tree implementation evaluates numeric thresholds, categorical variables like `Title` and `Embarked` are converted into binary indicator columns. Unlike linear models, keeping all $k$ categories causes no collinearity issues since trees evaluate features individually rather than solving a linear system.
* **No manual interaction terms:** A tree naturally captures interactions across features through sequential splits (e.g., splitting on `Pclass` and then `Sex`), making pre-engineered interaction columns redundant.
* **No mirrored binary pairs:** We include a single `Female` indicator rather than separate `Male` and `Female` columns.

In [32]:
train_tree = train.copy()
test_tree = test.copy()

for df in [train_tree, test_tree]:
    # Sex: single column is enough for a tree
    df['Female'] = (df['Sex'] == 'female').astype(int)

    # True categoricals: full one-hot (k-of-k) is fine for trees
    for level in ['S', 'C', 'Q']:
        df[f'Embarked_{level}'] = (df['Embarked'] == level).astype(int)

    for title in ['Mr', 'Miss', 'Mrs', 'Master', 'Rare']:
        df[f'Title_{title}'] = (df['Title'] == title).astype(int)

    for group, label in [('Solo', 'Solo'), ('Small_Group', 'Small'), ('Large_Group', 'Large')]:
        df[f'Ticket_{label}'] = (df['Ticket'] == group).astype(int)

    for group, label in [('Solo', 'Solo'), ('Small_Family', 'Small'), ('Large_Family', 'Large')]:
        df[f'Family_{label}'] = (df['Family_Group'] == group).astype(int)

    for deck in ['Unknown', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'T']:
        df[f'Cabin_{deck}'] = (df['Cabin'] == deck).astype(int)

drop_original = ['Sex', 'Embarked', 'Title', 'Ticket', 'Cabin', 'Family_Group']
for df in [train_tree, test_tree]:
    df.drop(columns=[c for c in drop_original if c in df.columns], inplace=True)

print(f"Tree feature table: {train_tree.shape[1]} columns (incl. Survived), no scaling applied")
print(train_tree.dtypes)

Tree feature table: 33 columns (incl. Survived), no scaling applied
Survived           int64
Pclass             int64
Age              float64
SibSp              int64
Parch              int64
Fare             float64
FamilySize         int64
IsChild            int64
WomanOrChild       int64
Female             int64
Embarked_S         int64
Embarked_C         int64
Embarked_Q         int64
Title_Mr           int64
Title_Miss         int64
Title_Mrs          int64
Title_Master       int64
Title_Rare         int64
Ticket_Solo        int64
Ticket_Small       int64
Ticket_Large       int64
Family_Solo        int64
Family_Small       int64
Family_Large       int64
Cabin_Unknown      int64
Cabin_A            int64
Cabin_B            int64
Cabin_C            int64
Cabin_D            int64
Cabin_E            int64
Cabin_F            int64
Cabin_G            int64
Cabin_T            int64
dtype: object


In [33]:
tree_features = [col for col in train_tree.columns if col != 'Survived']

y_train_tree = train_tree['Survived'].values
X_train_tree = train_tree[tree_features].values
X_test_tree = test_tree[tree_features].values

print(f"Tree feature count: {len(tree_features)}")

Tree feature count: 32


## Tree Models

### Single Decision Tree

We implement a recursive binary classification tree directly from our derived mathematical formulation:

1. **Gini Impurity:** Evaluates the uncertainty of a node sample set $\mathcal{S}$ using the binary Gini criterion:
   $$G(\mathcal{S}) = 2\hat{p}(1 - \hat{p}), \qquad \hat{p} = \frac{1}{|\mathcal{S}|}\sum_{i \in \mathcal{S}} y_i$$
2. **Split Optimization (`find_best_split`):** Searches all features $j$ and candidate thresholds $\tau$ to find the split $(j^*, \tau^*)$ maximizing Information Gain:
   $$\Delta(j, \tau) = G(\mathcal{S}) - \left(\frac{n_L}{n} G(\mathcal{S}_L) + \frac{n_R}{n} G(\mathcal{S}_R)\right)$$
3. **Recursive Partitioning (`build_tree`):** Recursively splits the dataset until reaching pure nodes ($G(\mathcal{S}) = 0$), the maximum tree depth (`max_depth`), or minimum node size (`min_samples`), assigning the majority class $c_\ell = \mathbf{1}[\hat{p} \ge 0.5]$ at leaf nodes.
4. **Inference (`predict_tree`):** Routes each test instance down decision boundaries to its corresponding leaf region $R_k$ to return prediction $T(x) = c_k$.

In [34]:
# 1. Node impurity (Definition 3.1) — Gini criterion
def gini(y):
    if len(y) == 0:
        return 0.0
    p = np.mean(y == 1)
    return 2.0 * p * (1.0 - p)

In [35]:
# 2. Split search — find (j*, tau*) maximizing information gain Delta(j, tau) (Definition 3.2)
def find_best_split(X, y):
    best_gain = -1.0
    best_feat, best_thresh = None, None
    n = len(y)
    parent_gini = gini(y)
    for feat_idx in range(X.shape[1]):
        col = X[:, feat_idx]
        for thresh in np.unique(col):
            left_mask = col <= thresh
            right_mask = ~left_mask
            # Skip invalid splits
            if np.sum(left_mask) == 0 or np.sum(right_mask) == 0:
                continue
            # Delta(j, tau) = H(S) - Q(j, tau), guaranteed non-negative by Proposition 3.1
            g_l, g_r = gini(y[left_mask]), gini(y[right_mask])
            child_gini = (np.sum(left_mask) / n) * g_l + (np.sum(right_mask) / n) * g_r
            gain = parent_gini - child_gini
            if gain > best_gain:
                best_gain, best_feat, best_thresh = gain, feat_idx, thresh
    return best_feat, best_thresh

In [36]:
# 3. Recursive tree construction (Definition 3.3), stored as nested dicts
def build_tree(X, y, depth=0, max_depth=5, min_samples=2):
    n_samples, n_labels = len(y), len(np.unique(y))
    # Evaluate stopping conditions -> declare a leaf, predict via majority vote
    if depth >= max_depth or n_labels <= 1 or n_samples < min_samples:
        return {'value': int(np.mean(y) >= 0.5)}
    feat, thresh = find_best_split(X, y)
    if feat is None:
        return {'value': int(np.mean(y) >= 0.5)}
    # Partition into S_L, S_R and recurse
    left_mask = X[:, feat] <= thresh
    return {
        'feature': feat,
        'threshold': thresh,
        'left': build_tree(X[left_mask], y[left_mask], depth + 1, max_depth, min_samples),
        'right': build_tree(X[~left_mask], y[~left_mask], depth + 1, max_depth, min_samples)
    }

In [37]:
# 4. Leaf prediction T(x) = c_k for the region R_k containing x
def predict_one(x, tree):
    if 'value' in tree:
        return tree['value']
    if x[tree['feature']] <= tree['threshold']:
        return predict_one(x, tree['left'])
    return predict_one(x, tree['right'])

def predict_tree(X, tree):
    return np.array([predict_one(row, tree) for row in X])

In [38]:
# 80/20 train-validation split
np.random.seed(42)
indices = np.random.permutation(len(X_train_tree))
split_idx = int(len(X_train_tree) * 0.8)
train_idx, val_idx = indices[:split_idx], indices[split_idx:]
X_tr, y_tr = X_train_tree[train_idx], y_train_tree[train_idx]
X_val, y_val = X_train_tree[val_idx], y_train_tree[val_idx]

# Grid over the two stopping-condition hyperparameters from Definition 3.3
depth_options = [3, 4, 5]
min_split_options = [10, 15, 20, 30]

best_tree_depth = None
best_min_samples = None
best_val_acc = -1.0

print("--- Grid Search: Finding Best Decision Tree Hyperparameters ---")
for depth in depth_options:
    for min_split in min_split_options:
        # build_tree() always uses the Gini criterion (Definition 3.1) --
        # this implementation has no criterion switch, unlike a class-based
        # version that would expose criterion='gini'/'entropy'
        tree = build_tree(X_tr, y_tr, depth=0, max_depth=depth, min_samples=min_split)
        val_preds = predict_tree(X_val, tree)
        val_acc = np.mean(val_preds == y_val) * 100
        print(f"Max Depth: {depth:<2} | Min Samples Split: {min_split:<2} | Val Accuracy: {val_acc:.2f}%")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_tree_depth = depth
            best_min_samples = min_split

print(f"\n Best Parameters Found -> max_depth: {best_tree_depth}, min_samples_split: {best_min_samples} (Val Accuracy: {best_val_acc:.2f}%)")

--- Grid Search: Finding Best Decision Tree Hyperparameters ---
Max Depth: 3  | Min Samples Split: 10 | Val Accuracy: 81.01%
Max Depth: 3  | Min Samples Split: 15 | Val Accuracy: 81.01%
Max Depth: 3  | Min Samples Split: 20 | Val Accuracy: 81.01%
Max Depth: 3  | Min Samples Split: 30 | Val Accuracy: 81.01%
Max Depth: 4  | Min Samples Split: 10 | Val Accuracy: 81.56%
Max Depth: 4  | Min Samples Split: 15 | Val Accuracy: 81.56%
Max Depth: 4  | Min Samples Split: 20 | Val Accuracy: 81.56%
Max Depth: 4  | Min Samples Split: 30 | Val Accuracy: 81.01%
Max Depth: 5  | Min Samples Split: 10 | Val Accuracy: 81.01%
Max Depth: 5  | Min Samples Split: 15 | Val Accuracy: 81.01%
Max Depth: 5  | Min Samples Split: 20 | Val Accuracy: 82.12%
Max Depth: 5  | Min Samples Split: 30 | Val Accuracy: 80.45%

 Best Parameters Found -> max_depth: 5, min_samples_split: 20 (Val Accuracy: 82.12%)


In [39]:
# Retrain on the FULL training set using best parameters
final_tree = build_tree(X_train_tree, y_train_tree, depth=0, max_depth=best_tree_depth, min_samples=best_min_samples)

train_preds_tree = predict_tree(X_train_tree, final_tree)
acc_tree = np.mean(train_preds_tree == y_train_tree) * 100
print(f"Final Decision Tree Training Accuracy: {acc_tree:.2f}%")

test_preds_tree = predict_tree(X_test_tree, final_tree)

submission_tree = pd.DataFrame({
    'PassengerId': test_passenger_ids,
    'Survived': test_preds_tree
})
submission_tree.to_csv('submission_decision_tree.csv', index=False)

print("\n Predictions successfully exported to 'submission_decision_tree.csv'!")
print(f"Total Test Passengers: {len(submission_tree)}")
print(f"Predicted Survivors: {submission_tree['Survived'].sum()}")

Final Decision Tree Training Accuracy: 84.96%

 Predictions successfully exported to 'submission_decision_tree.csv'!
Total Test Passengers: 418
Predicted Survivors: 125


### Single Decision Tree Feature Importance

We evaluate feature importance by traversing the trained decision tree and counting how many times each feature $j$ was selected as the optimal split variable ($j^*$). Features chosen more frequently across decision nodes serve as primary partition boundaries for predicting survival.

In [40]:
# Feature importance: count how many times each feature was chosen as a
# split (Definition 3.2's j* winner), tallied across the whole tree
def count_tree_splits(node, counts):
    if node is None or 'value' in node:
        return
    counts[node['feature']] += 1
    count_tree_splits(node['left'], counts)
    count_tree_splits(node['right'], counts)

tree_counts = np.zeros(len(tree_features))
count_tree_splits(final_tree, tree_counts)

dt_importance_df = pd.DataFrame({
    'Feature': tree_features,
    'Tree_Splits': tree_counts
}).sort_values(by='Tree_Splits', ascending=False)

print("--- Decision Tree: Feature Importance (Split Counts) ---")
dt_importance_df.head(10)

--- Decision Tree: Feature Importance (Split Counts) ---


,Feature,Tree_Splits
1,Age,6.0
4,Fare,5.0
0,Pclass,2.0
19,Ticket_Large,2.0
5,FamilySize,1.0
28,Cabin_E,1.0
7,WomanOrChild,1.0
23,Cabin_Unknown,1.0
11,Embarked_Q,1.0
24,Cabin_A,0.0


### Random Forest

We implement a Random Forest ensemble directly from the derived mathematical results to decorrelate individual trees and reduce prediction variance:

1. **Bootstrap Sampling (`bootstrap_sample`):** Draws $n$ observations uniformly with replacement ($\mathcal{D}^{(b)}$) for each tree, where each observation has an inclusion probability of $1 - e^{-1} \approx 63.2\%$.
2. **Random Feature Subspace Split (`find_best_split_rf`):** Restricts the split candidate search to a randomly chosen subset of $m_{\text{try}} = \lfloor \sqrt{p} \rfloor$ features at each split step:
   $$(j^*, \tau^*) = \operatorname*{argmax}_{j \in \mathcal{M}, \, \tau} \Delta(j, \tau), \qquad |\mathcal{M}| = m_{\text{try}}$$
3. **Ensemble Construction (`build_forest`):** Recursively trains $N$ decorrelated decision trees on independent bootstrap samples.
4. **Majority Vote Aggregation (`predict_forest`):** Combines the predictions of all $N$ trees via empirical majority voting:
   $$\hat{y}(x) = \mathbf{1}\!\left[\frac{1}{N}\sum_{b=1}^N T_b(x) \ge 0.5\right]$$

In [41]:
# Split search restricted to a random feature subset (Definition 3.5, step 2:
# "randomly select a subset of m_try = floor(sqrt(p)) features out of all p
# features, searching for the optimal split (j*, tau*) within this sampled
# feature set"). Otherwise identical to find_best_split's Delta(j, tau)
# search (Definition 3.2).
def find_best_split_rf(X, y, n_features_subsample):
    best_gain = -1.0
    best_feat, best_thresh = None, None
    n = len(y)
    parent_gini = gini(y)  # reuses the same gini() from the tree section

    feature_indices = np.random.choice(X.shape[1], n_features_subsample, replace=False)
    for feat_idx in feature_indices:
        col = X[:, feat_idx]
        for thresh in np.unique(col):
            left_mask = col <= thresh
            right_mask = ~left_mask
            if np.sum(left_mask) == 0 or np.sum(right_mask) == 0:
                continue
            g_l, g_r = gini(y[left_mask]), gini(y[right_mask])
            child_gini = (np.sum(left_mask) / n) * g_l + (np.sum(right_mask) / n) * g_r
            gain = parent_gini - child_gini
            if gain > best_gain:
                best_gain, best_feat, best_thresh = gain, feat_idx, thresh
    return best_feat, best_thresh

In [42]:
# Recursive tree construction (Definition 3.3) -- same stopping conditions
# and S_L/S_R partitioning as build_tree(), but every node calls
# find_best_split_rf() instead, so m_try applies at every single split
# down the tree, not just once per tree.
def build_tree_rf(X, y, depth, max_depth, min_samples, n_features_subsample):
    n_samples, n_labels = len(y), len(np.unique(y))
    if depth >= max_depth or n_labels <= 1 or n_samples < min_samples:
        return {'value': int(np.mean(y) >= 0.5)}
    feat, thresh = find_best_split_rf(X, y, n_features_subsample)
    if feat is None:
        return {'value': int(np.mean(y) >= 0.5)}
    left_mask = X[:, feat] <= thresh
    return {
        'feature': feat,
        'threshold': thresh,
        'left': build_tree_rf(X[left_mask], y[left_mask], depth + 1, max_depth, min_samples, n_features_subsample),
        'right': build_tree_rf(X[~left_mask], y[~left_mask], depth + 1, max_depth, min_samples, n_features_subsample)
    }

In [43]:
# Bootstrap sample (Definition 3.4): draw n samples from D at random WITH
# replacement to build D^(b) for one tree. Duplicates some passengers,
# omits others entirely -- each passenger has ~63.2% odds of appearing in
# any given D^(b) (Lemma 3.1).
def bootstrap_sample(X, y):
    n_samples = X.shape[0]
    indices = np.random.choice(n_samples, n_samples, replace=True)
    return X[indices], y[indices]

In [44]:
# Random Forest ensemble (Definition 3.5): construct N trees {T_1, ..., T_N},
# each independently on its own bootstrap sample D^(b) with splits
# restricted to m_try = floor(sqrt(p)) random features.
def build_forest(X, y, n_trees=100, max_depth=5, min_samples=2, m_try=None):
    n_features = X.shape[1]
    if m_try is None:
        m_try = int(np.floor(np.sqrt(n_features)))

    trees = []
    for _ in range(n_trees):
        X_sample, y_sample = bootstrap_sample(X, y)
        tree = build_tree_rf(X_sample, y_sample, depth=0, max_depth=max_depth,
                              min_samples=min_samples, n_features_subsample=m_try)
        trees.append(tree)
    return trees

In [45]:
# Ensemble prediction (Definition 3.5): y_hat(x) = 1[(1/N) * sum_b T_b(x) >= 1/2].
# Reuses predict_tree() from the single-tree section unchanged -- a forest
# tree is stored in the exact same dict shape as a single tree, so no
# separate predict_tree_rf() is needed.
def predict_forest(X, trees):
    tree_preds = np.array([predict_tree(X, tree) for tree in trees])
    majority_votes = np.mean(tree_preds, axis=0) >= 0.5
    return majority_votes.astype(int)

In [46]:
np.random.seed(42)
indices = np.random.permutation(len(X_train_tree))
split_idx = int(len(X_train_tree) * 0.8)
train_idx, val_idx = indices[:split_idx], indices[split_idx:]
X_tr, y_tr = X_train_tree[train_idx], y_train_tree[train_idx]
X_val, y_val = X_train_tree[val_idx], y_train_tree[val_idx]

depth_options = [3, 4, 5]
n_trees_options = [100, 200, 300]

best_rf_depth = None
best_n_trees = None
best_val_acc_rf = -1.0

print("--- Grid Search: Finding Best Random Forest Hyperparameters ---")
for depth in depth_options:
    for n_trees in n_trees_options:
        forest = build_forest(X_tr, y_tr, n_trees=n_trees, max_depth=depth, min_samples=10)
        val_preds = predict_forest(X_val, forest)
        val_acc = np.mean(val_preds == y_val) * 100
        print(f"Max Depth: {depth:<2} | N Trees: {n_trees:<3} | Val Accuracy: {val_acc:.2f}%")
        if val_acc > best_val_acc_rf:
            best_val_acc_rf = val_acc
            best_rf_depth = depth
            best_n_trees = n_trees

print(f"\n Best Parameters Found -> max_depth: {best_rf_depth}, n_trees: {best_n_trees} (Val Accuracy: {best_val_acc_rf:.2f}%)")

--- Grid Search: Finding Best Random Forest Hyperparameters ---
Max Depth: 3  | N Trees: 100 | Val Accuracy: 83.80%
Max Depth: 3  | N Trees: 200 | Val Accuracy: 83.80%
Max Depth: 3  | N Trees: 300 | Val Accuracy: 83.80%
Max Depth: 4  | N Trees: 100 | Val Accuracy: 83.80%
Max Depth: 4  | N Trees: 200 | Val Accuracy: 83.80%
Max Depth: 4  | N Trees: 300 | Val Accuracy: 83.80%
Max Depth: 5  | N Trees: 100 | Val Accuracy: 83.80%
Max Depth: 5  | N Trees: 200 | Val Accuracy: 83.80%
Max Depth: 5  | N Trees: 300 | Val Accuracy: 83.80%

 Best Parameters Found -> max_depth: 3, n_trees: 100 (Val Accuracy: 83.80%)


In [47]:
# Retrain on the FULL training set using best hyperparameters
np.random.seed(42)
final_forest = build_forest(X_train_tree, y_train_tree, n_trees=best_n_trees, max_depth=best_rf_depth, min_samples=5)

train_preds_rf = predict_forest(X_train_tree, final_forest)
acc_rf = np.mean(train_preds_rf == y_train_tree) * 100
print(f"Final Random Forest Training Accuracy: {acc_rf:.2f}%")

test_preds_rf = predict_forest(X_test_tree, final_forest)
submission_rf = pd.DataFrame({
    'PassengerId': test_passenger_ids,
    'Survived': test_preds_rf
})
submission_rf.to_csv('submission_random_forest.csv', index=False)

print("\n Predictions successfully exported to 'submission_random_forest.csv'!")
print(f"Total Test Passengers: {len(submission_rf)}")
print(f"Predicted Survivors: {submission_rf['Survived'].sum()}")

Final Random Forest Training Accuracy: 83.50%

 Predictions successfully exported to 'submission_random_forest.csv'!
Total Test Passengers: 418
Predicted Survivors: 163


### Random Forest Feature Importance

We evaluate the relative importance of each feature by measuring its total split frequency across the ensemble (`RF_Importance`). This tallies how often a feature is chosen as the optimal split across all $N=100$ trees, divided by the total split count in the forest. 

Even though candidate splits are evaluated from a random subset of size $m_{\text{try}} = \lfloor \sqrt{p} \rfloor$ at each node, strong predictors consistently maximize Information Gain and win the split whenever sampled, directly reflecting their predictive power.

In [50]:
# 1. Fit the final Random Forest on the full training data
final_rf = build_forest(X_train_tree, y_train_tree, n_trees=100, max_depth=3, min_samples=10)

# 2. Initialize feature list and split counter
tree_features = [col for col in train_tree.columns if col != 'Survived']
rf_counts = np.zeros(len(tree_features))

# 3. Traverse all trees across the ensemble and count feature splits
def count_rf_splits(tree):
    if tree is None or 'feature' not in tree:
        return
    rf_counts[tree['feature']] += 1
    count_rf_splits(tree.get('left'))
    count_rf_splits(tree.get('right'))

for tree in final_rf:
    count_rf_splits(tree)

# 4. Normalize split counts to get relative importance percentages
total_rf_splits = rf_counts.sum()
rf_importance = (rf_counts / total_rf_splits) if total_rf_splits > 0 else rf_counts

rf_importance_df = pd.DataFrame({
    'Feature': tree_features,
    'RF_Splits': rf_counts.astype(int),
    'RF_Importance': rf_importance
})

# 5. Sort by importance and display top 10
rf_importance_df = rf_importance_df.sort_values(by='RF_Importance', ascending=False).reset_index(drop=True)

print("=== Random Forest Feature Importance ===")
display(rf_importance_df.head(10))

=== Random Forest Feature Importance ===


,Feature,RF_Splits,RF_Importance
0,Pclass,56,0.082353
1,Fare,56,0.082353
2,WomanOrChild,52,0.076471
3,Cabin_Unknown,50,0.073529
4,FamilySize,40,0.058824
5,Female,33,0.048529
6,Title_Mr,31,0.045588
7,Title_Miss,30,0.044118
8,Age,29,0.042647
9,Family_Small,27,0.039706


## Summary: Tree Model Results

* **Single Tree:** $84.96\%$ train accuracy $\rightarrow$ **$0.75358$** test score ($125$ survivors predicted). Prone to single-tree sample variance.
* **Random Forest:** $83.50\%$ train accuracy $\rightarrow$ **$0.77751$** test score ($163$ survivors predicted). Feature subsampling ($m_{\text{try}} = 5$) successfully stabilized predictions across the ensemble.
* **Core Predictors:** Model splits are predominantly driven by `Fare`, `Pclass`, and `WomanOrChild`.

## 10. Final Synthesis: Comparing All Models

In [ ]:
sub_logreg = pd.read_csv('submission_logreg.csv')
sub_tree = pd.read_csv('submission_decision_tree.csv')
sub_rf = pd.read_csv('submission_random_forest.csv')

print("--- Model Comparison: Predicted Survivors ---")
print(f"Logistic Regression (Unreg): {sub_logreg['Survived'].sum()} / {len(sub_logreg)}")
print(f"Single Decision Tree:        {sub_tree['Survived'].sum()} / {len(sub_tree)}")
print(f"Random Forest:                {sub_rf['Survived'].sum()} / {len(sub_rf)}")

agree_rf_logreg = np.mean(sub_rf['Survived'] == sub_logreg['Survived']) * 100
agree_rf_tree = np.mean(sub_rf['Survived'] == sub_tree['Survived']) * 100

print("\n--- Model Agreement Check ---")
print(f"Random Forest vs. Logistic Regression match: {agree_rf_logreg:.2f}% of the time")
print(f"Random Forest vs. Single Decision Tree match: {agree_rf_tree:.2f}% of the time")

### What Drove the Predictions?

#### 1. Evacuation Priority & Demographics
* **Logistic Regression:** Led by composite evacuation indicators: `WomanOrChild` ($w = +1.245$), `Female` ($w = +0.977$), and `Title_Mrs` ($w = +0.640$).
* **Tree Models:** `WomanOrChild` was the 3rd most frequent split in the Random Forest (**7.65%** importance), while `Female` (**4.85%**) and `Age` (**4.26%**) served as key secondary splitters.

#### 2. Socio-Economic Status
* `Pclass` and `Fare` tied as the top split drivers across the entire Random Forest (**8.24%** importance each).
* `Pclass` also carried the single largest negative coefficient in Logistic Regression ($w = -0.763$).

#### 3. Party Size & Cabin Visibility
* **Group Size Dynamics:** Large parties were penalized in Logistic Regression (`Ticket_Large` $w = -0.494$, `Family_Large` $w = -0.452$), while `FamilySize` ranked 5th in tree importance (**5.88%**).
* **Missing Cabin Indicator:** Having an unrecorded deck (`Cabin_Unknown`) was the 4th strongest tree split (**7.35%** importance), effectively separating 3rd-class steerage from premium deck holders.

#### 4. Model Concordance and Leaderboard Alignment
* The top two Kaggle performers, **Random Forest (77.75%)** and **Unregularized Logistic Regression (77.51%)**, matched on **97.85%** of all test predictions (differing on only 9 out of 418 passengers), confirming that both models converged on the same core decision boundary.

---
#### Notes
* **Core Analysis:** All feature engineering, mathematical derivations, model training, and evaluation were developed by the author.
* **AI Assistance:** Claude and Gemini were consulted for code verification, debugging checks, text refinement, and advice on code presentation.